# Perfil de Menor Risco em Pacientes Documentadas

**Pergunta:** Entre pacientes com documentação clínica registrada,
em quais condições o risco de retorno em 30 dias é MENOR?

## Lógica da análise

Três camadas:
1. **Descritiva** — comparar taxa de retorno 30d por condição clínica
   dentro do subgrupo documentado vs não documentado
2. **Preditiva** — SHAP "reverso": quais features empurram o score para BAIXO
3. **Perfil protetor** — combinar condições que co-ocorrem em baixo risco

Hipóteses clínicas testadas:
- Gestação de termo (GUTT) + primigesta → menor risco
- Documentação com clinical_status="stable" → menos retorno
- IG adequada (37-40s) + peso RN normal → menos retorno
- Pré-natal completo (NR_CONSULTAS ≥ 6) → menos retorno
- Baixo escore Manchester + sem red_flag → menos retorno

In [ ]:
from __future__ import annotations
import sys, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import duckdb

warnings.filterwarnings("ignore")

ROOT     = Path(__file__).resolve().parent.parent
PARQUET  = ROOT / "data" / "gestante_quadro_clinico.parquet"
LLM_PAR  = ROOT / "out"  / "llm_extracted_20k.parquet"
OUT_DIR  = ROOT / "out"  / f"menor_risco_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "src"))

## 1. Carga e definição dos subgrupos

In [ ]:
import duckdb as _ddb
_DUCKDB        = ROOT / "data" / "gestante.duckdb"
_PARQUET_ENRIQ = ROOT / "data" / "gestante_quadro_clinico_enriquecido.parquet"

_tabs = _ddb.connect(str(_DUCKDB)).execute("SHOW TABLES").df()["name"].tolist()
if "gestante_enriquecido" in _tabs:
    _con = _ddb.connect(str(_DUCKDB))
    df = _con.execute("SELECT * FROM gestante_enriquecido").df()
    _con.close()
    df.columns = df.columns.str.lower()
    print(f"DuckDB gestante_enriquecido carregado: {len(df):,} registros")
elif _PARQUET_ENRIQ.exists():
    df = pd.read_parquet(_PARQUET_ENRIQ)
    df.columns = df.columns.str.lower()
    print(f"Parquet enriquecido carregado: {len(df):,} registros")
else:
    # fallback: processa do zero
    import sys; sys.path.insert(0, str(ROOT / "src"))
    from build_obstetric_features import enrich_obstetric
    df = pd.read_parquet(PARQUET)  # use whatever PARQUET variable exists in the file
    df.columns = df.columns.str.lower()
    df = df[df["is_recem_nascido"] == 0].copy()
    LLM_PARQUET = ROOT / "out" / "llm_extracted_20k.parquet"
    df = enrich_obstetric(df, llm_parquet=str(_DUCKDB))
    df = df[(df.flag_neonatal == 0)].copy()
    print(f"Processado do zero: {len(df):,} registros")

# Target
df["y"] = (df["sn_retorno_30d"].fillna("N") == "S").astype(int)

# Flags de documentação
df["tem_evolucao"]  = df["qtd_evolucoes_documentadas"].fillna(0).gt(0).astype(int)
df["tem_queixa"]    = df["ds_queixa_principal"].notna().astype(int)
df["tem_qualquer"]  = ((df["tem_evolucao"] == 1) | (df["tem_queixa"] == 1)).astype(int)
df["doc_rica"]      = (df["qtd_evolucoes_documentadas"].fillna(0) >= 3).astype(int)

# Tipos numéricos
for c in ["semanas_gestacao","idade_anos","vl_escore_triagem",
          "nr_consultas_pre_natal","duracao_horas_aprox",
          "media_peso_rn_g","min_apgar_5min","qtd_pre_med"]:
    df[c] = pd.to_numeric(df.get(c), errors="coerce")

# Segmento CID
def _seg(c):
    if pd.isna(c): return "SEM_CID"
    c = str(c).upper()
    if c[:3] in [f"O{i:02d}" for i in range(10,17)]: return "HIPERTENSIVOS"
    if c[:3] == "O24": return "DIABETES_GEST"
    if c[:3] in ["O60","O42","O47"]: return "PARTO_PREMATURO"
    if c[0]=="O" and c[1] in "234": return "HEMORRAGIA"
    if c[:3] in ["O98","O99"]: return "COMORBIDADES"
    if c[0]=="O": return "OUTROS_OBST"
    if c[:2]=="Z3": return "PRE_NATAL"
    return "OUTROS"

df["segmento"] = df["cd_cid"].apply(_seg)

n_total = len(df)
n_doc   = df["tem_qualquer"].sum()
n_evol  = df["tem_evolucao"].sum()
print(f"Gestantes total       : {n_total:,}")
print(f"Com qualquer texto    : {n_doc:,} ({n_doc/n_total*100:.1f}%)")
print(f"Com evolução médica   : {n_evol:,} ({n_evol/n_total*100:.1f}%)")
print(f"Retorno 30d (total)   : {df['y'].sum():,} ({df['y'].mean()*100:.1f}%)")
print(f"Retorno 30d (doc)     : {df[df['tem_qualquer']==1]['y'].mean()*100:.1f}%")
print(f"Retorno 30d (sem doc) : {df[df['tem_qualquer']==0]['y'].mean()*100:.1f}%")

## 2. Comparação Documentadas × Não Documentadas por Condição Clínica

In [ ]:
resultados = []

for seg in df["segmento"].value_counts().index:
    sub     = df[df["segmento"] == seg]
    sub_doc = sub[sub["tem_qualquer"] == 1]
    sub_nd  = sub[sub["tem_qualquer"] == 0]

    if len(sub_doc) < 20 or len(sub_nd) < 20:
        continue

    resultados.append({
        "segmento"       : seg,
        "n_total"        : len(sub),
        "n_doc"          : len(sub_doc),
        "n_sem_doc"      : len(sub_nd),
        "ret_doc_pct"    : round(sub_doc["y"].mean() * 100, 2),
        "ret_sem_doc_pct": round(sub_nd["y"].mean() * 100, 2),
        "delta_pp"       : round((sub_doc["y"].mean() - sub_nd["y"].mean()) * 100, 2),
    })

comp = pd.DataFrame(resultados).sort_values("ret_doc_pct")
print("\nRetorno 30d: Documentadas vs Não Documentadas por Segmento CID")
print("(delta_pp < 0 = documentadas têm MENOS retorno)")
print(comp[["segmento","n_doc","ret_doc_pct","ret_sem_doc_pct","delta_pp"]].to_string(index=False))
comp.to_csv(OUT_DIR / "doc_vs_nao_doc_por_segmento.csv", index=False)

# Gráfico
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(comp))
w = 0.35
ax.bar([i - w/2 for i in x], comp["ret_doc_pct"],    w, label="Documentadas",     color="steelblue", alpha=0.85)
ax.bar([i + w/2 for i in x], comp["ret_sem_doc_pct"],w, label="Sem documentação",  color="lightcoral",alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(comp["segmento"], rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Taxa de retorno 30d (%)")
ax.set_title("Retorno 30d: Documentadas × Não Documentadas por Segmento CID")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "01_doc_vs_nao_doc.png", dpi=150)
plt.show(); plt.close()

## 3. Perfil Protetor — Dentro das Documentadas

In [ ]:
# Foco: pacientes COM documentação — quais condições se associam a MENOS retorno?
df_doc = df[df["tem_qualquer"] == 1].copy()

print(f"\nSubgrupo documentadas: {len(df_doc):,}")
print(f"Retorno 30d neste subgrupo: {df_doc['y'].mean()*100:.1f}%")

# Condições candidatas a perfil protetor
condicoes = {}

# 1. Tipo de gestação
if "tipo_gestacao" in df_doc.columns:
    for tipo in ["GUTT","GURT","GUPT","GGUT"]:
        mask = df_doc["tipo_gestacao"] == tipo
        if mask.sum() >= 20:
            condicoes[f"tipo={tipo}"] = df_doc[mask]["y"].mean() * 100

# 2. Perfil obstétrico
if "perfil_historico" in df_doc.columns:
    for perfil in df_doc["perfil_historico"].dropna().unique():
        mask = df_doc["perfil_historico"] == perfil
        if mask.sum() >= 20:
            condicoes[f"perfil={perfil}"] = df_doc[mask]["y"].mean() * 100

# 3. IG adequada (37-40s)
ig = pd.to_numeric(df_doc.get("ig_obs", df_doc.get("semanas_gestacao")), errors="coerce")
mask_ig = ig.between(37, 40, inclusive="both")
if mask_ig.sum() >= 20:
    condicoes["IG 37-40s (termo)"] = df_doc[mask_ig]["y"].mean() * 100

# 4. Sem red_flag (LLM)
if "red_flag" in df_doc.columns:
    rf = pd.to_numeric(df_doc["red_flag"], errors="coerce").fillna(-1)
    mask_rf = rf == 0
    if mask_rf.sum() >= 20:
        condicoes["Sem red_flag (LLM)"] = df_doc[mask_rf]["y"].mean() * 100
    mask_rfs = rf == 1
    if mask_rfs.sum() >= 20:
        condicoes["Com red_flag (LLM)"] = df_doc[mask_rfs]["y"].mean() * 100

# 5. clinical_status estável
if "clinical_status" in df_doc.columns:
    for status in ["stable","improving","worsening","critical"]:
        mask_s = df_doc["clinical_status"].fillna("") == status
        if mask_s.sum() >= 20:
            condicoes[f"status={status}"] = df_doc[mask_s]["y"].mean() * 100

# 6. Severity LLM
if "severity" in df_doc.columns:
    for sev in ["low","moderate","high","critical"]:
        mask_sv = df_doc["severity"].fillna("") == sev
        if mask_sv.sum() >= 20:
            condicoes[f"severity={sev}"] = df_doc[mask_sv]["y"].mean() * 100

# 7. Manchester baixo (≤2 = verde/amarelo)
if "cd_cor_manchester" in df_doc.columns:
    mcr = pd.to_numeric(df_doc["cd_cor_manchester"], errors="coerce")
    for faixa, lbl in [((None,2),"Manchester verde/amarelo"),((3,None),"Manchester laranja/vermelho")]:
        lo, hi = faixa
        if lo is None:
            mask_m = mcr <= hi
        elif hi is None:
            mask_m = mcr >= lo
        else:
            mask_m = mcr.between(lo, hi)
        if mask_m.sum() >= 20:
            condicoes[lbl] = df_doc[mask_m]["y"].mean() * 100

# 8. Pré-natal adequado
if "nr_consultas_pre_natal" in df_doc.columns:
    nc = pd.to_numeric(df_doc["nr_consultas_pre_natal"], errors="coerce")
    mask_pn = nc >= 6
    if mask_pn.sum() >= 20:
        condicoes["Pré-natal ≥ 6 consultas"] = df_doc[mask_pn]["y"].mean() * 100

# 9. Primigesta + termo
mask_pgt = (
    (df_doc.get("primigesta", pd.Series([0]*len(df_doc), index=df_doc.index)).fillna(0) == 1) &
    (ig.reindex(df_doc.index, fill_value=np.nan).between(37, 40))
)
if mask_pgt.sum() >= 20:
    condicoes["Primigesta + IG 37-40s"] = df_doc[mask_pgt]["y"].mean() * 100

# Referência: taxa geral nos documentados
ref = df_doc["y"].mean() * 100
condicoes["─── MÉDIA GERAL ───"] = ref

# Organiza resultado
cond_df = (pd.DataFrame.from_dict(condicoes, orient="index", columns=["ret_pct"])
             .reset_index().rename(columns={"index":"condicao"})
             .sort_values("ret_pct"))

cond_df["delta_vs_media"] = (cond_df["ret_pct"] - ref).round(2)
cond_df["perfil"] = cond_df["ret_pct"].apply(
    lambda v: "PROTETOR" if v < ref - 2 else ("RISCO" if v > ref + 2 else "NEUTRO")
)

print("\n" + "="*65)
print("PERFIS DE MENOR RISCO — Pacientes Documentadas")
print(f"Referência (média geral): {ref:.1f}%")
print("="*65)
print(cond_df[["condicao","ret_pct","delta_vs_media","perfil"]].to_string(index=False))
cond_df.to_csv(OUT_DIR / "perfil_protetor_documentadas.csv", index=False)

# Gráfico
fig, ax = plt.subplots(figsize=(10, max(6, len(cond_df) * 0.4)))
cores = {"PROTETOR":"#2ca02c","NEUTRO":"steelblue","RISCO":"#d62728"}
for i, (_, row) in enumerate(cond_df.iterrows()):
    cor = cores.get(row["perfil"], "gray")
    ax.barh(row["condicao"], row["ret_pct"], color=cor, alpha=0.85)
ax.axvline(ref, color="black", ls="--", lw=1.5, label=f"Média ({ref:.1f}%)")
ax.set_xlabel("Taxa de retorno 30d (%)")
ax.set_title("Perfil de Risco por Condição — Pacientes COM Documentação Clínica\n(verde=protetor, vermelho=risco)")
ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PLOT_DIR / "02_perfil_protetor.png", dpi=150)
plt.show(); plt.close()

## 4. SHAP Inverso — Features que Reduzem o Risco

In [ ]:
try:
    import shap
    from sklearn.compose import ColumnTransformer
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from lightgbm import LGBMClassifier

    FEATURES = [f for f in [
        "segmento","tipo_gestacao","perfil_historico",
        "idade_anos","ig_obs","semanas_gestacao",
        "vl_escore_triagem","cd_cor_manchester",
        "qtd_pre_med","qtd_evolucoes_documentadas",
        "duracao_horas_aprox","media_peso_rn_g",
        "risco_gestacao","gestacao_pretermo","primigesta",
        "grande_multipara_v2","tem_aborto_v2","aborto_recorrente",
        "red_flag","severity_score","clinical_status_score",
    ] if f in df_doc.columns]

    X = df_doc[FEATURES]
    y = df_doc["y"]

    num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in FEATURES if c not in num_cols]

    pre = ColumnTransformer([
        ("num", Pipeline([("i",SimpleImputer(strategy="median")),
                          ("s",StandardScaler())]), num_cols),
        ("cat", Pipeline([("i",SimpleImputer(strategy="most_frequent")),
                          ("o",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]), cat_cols),
    ], remainder="drop")

    X_t = pre.fit_transform(X)
    clf = LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                         n_jobs=-1, random_state=42, verbose=-1)
    clf.fit(X_t, y)

    feat_names = [n.split("__",1)[-1] for n in pre.get_feature_names_out()]
    rng = np.random.default_rng(42)
    idx_s = rng.integers(0, len(X_t), min(800, len(X_t)))
    X_s  = np.asarray(X_t[idx_s])

    explainer  = shap.TreeExplainer(clf)
    sv = explainer.shap_values(X_s)
    if isinstance(sv, list): sv = sv[1]
    if getattr(sv,"ndim",0)==3: sv = sv[:,:,1]
    sv = np.asarray(sv)

    # Features que REDUZEM o risco (SHAP negativo médio)
    shap_mean = sv.mean(axis=0)   # média (com sinal): negativo = protetor
    shap_abs  = np.abs(sv).mean(axis=0)

    shap_df = pd.DataFrame({
        "feature"      : feat_names,
        "shap_medio"   : shap_mean,    # negativo = protetor
        "shap_abs"     : shap_abs,
        "direcao"      : ["PROTETOR" if v < -0.005 else
                          "RISCO"    if v >  0.005 else "NEUTRO"
                          for v in shap_mean],
    }).sort_values("shap_medio")

    protetores = shap_df[shap_df["direcao"]=="PROTETOR"].head(15)
    risco_feat = shap_df[shap_df["direcao"]=="RISCO"].head(10)

    print("\n" + "="*60)
    print("TOP FEATURES PROTETORAS (SHAP médio negativo)")
    print("="*60)
    print(protetores[["feature","shap_medio","shap_abs"]].to_string(index=False))
    print("\nTOP FEATURES DE RISCO (SHAP médio positivo)")
    print(risco_feat[["feature","shap_medio","shap_abs"]].to_string(index=False))

    # Gráfico bidirecional
    fig, ax = plt.subplots(figsize=(10, 8))
    top_n = pd.concat([protetores.head(10), risco_feat.head(8)]).sort_values("shap_medio")
    cores_s = ["#2ca02c" if v < 0 else "#d62728" for v in top_n["shap_medio"]]
    ax.barh(top_n["feature"], top_n["shap_medio"], color=cores_s, alpha=0.85)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("SHAP médio (negativo = protetor, positivo = risco)")
    ax.set_title("Features Protetoras e de Risco — Pacientes COM Documentação\n(LightGBM, SHAP TreeExplainer)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "03_shap_protetor_risco.png", dpi=150)
    plt.show(); plt.close()

    shap_df.to_csv(OUT_DIR / "shap_protetor_risco.csv", index=False)

except ImportError as e:
    print(f"SHAP/LightGBM não disponível: {e}")

## 5. Combinações Protetoras — Perfil Composto de Menor Risco

In [ ]:
print("\n" + "="*65)
print("COMBINAÇÕES PROTETORAS — menor risco com documentação")
print("="*65)

combos = {
    # Condição                                      Máscara
    "GUTT + sem red_flag":
        (df_doc.get("tipo_gestacao","") == "GUTT") &
        (pd.to_numeric(df_doc.get("red_flag",0), errors="coerce").fillna(1) == 0),

    "Primigesta + IG 37-40s + sem red_flag":
        (df_doc.get("primigesta",pd.Series([0]*len(df_doc),index=df_doc.index)).fillna(0)==1) &
        (pd.to_numeric(df_doc.get("ig_obs",df_doc.get("semanas_gestacao",np.nan)),
                       errors="coerce").between(37,40)) &
        (pd.to_numeric(df_doc.get("red_flag",0),errors="coerce").fillna(1)==0),

    "Manchester ≤ 2 + sem red_flag + status=stable":
        (pd.to_numeric(df_doc.get("cd_cor_manchester",99),errors="coerce").fillna(99)<=2) &
        (pd.to_numeric(df_doc.get("red_flag",0),errors="coerce").fillna(1)==0) &
        (df_doc.get("clinical_status","").fillna("")=="stable"),

    "MULTIPARA_SEM_PERDA + termo + Manchester baixo":
        (df_doc.get("perfil_historico","").fillna("")=="MULTIPARA_SEM_PERDA") &
        (pd.to_numeric(df_doc.get("semanas_gestacao",0),errors="coerce").fillna(0)>=37) &
        (pd.to_numeric(df_doc.get("cd_cor_manchester",99),errors="coerce").fillna(99)<=2),

    "Documentação rica (≥3 evoluções) + severity=low":
        (df_doc.get("qtd_evolucoes_documentadas",0).fillna(0)>=3) &
        (df_doc.get("severity","").fillna("")=="low"),
}

rows_combo = []
for nome, mask in combos.items():
    try:
        sub_c = df_doc[mask.values if hasattr(mask,"values") else mask]
        if len(sub_c) < 10:
            print(f"  {nome}: n={len(sub_c)} — insuficiente")
            continue
        ret = sub_c["y"].mean() * 100
        rows_combo.append({
            "combinacao"     : nome,
            "n"              : len(sub_c),
            "ret_pct"        : round(ret, 2),
            "delta_vs_media" : round(ret - ref, 2),
        })
        sinal = "✅ PROTETOR" if ret < ref - 2 else ("⚠️  RISCO" if ret > ref + 2 else "~neutro")
        print(f"  {sinal}  {nome}")
        print(f"          n={len(sub_c):,}  retorno={ret:.1f}%  (média={ref:.1f}%)")
    except Exception as e:
        print(f"  SKIP {nome}: {e}")

if rows_combo:
    combo_df = pd.DataFrame(rows_combo).sort_values("ret_pct")
    combo_df.to_csv(OUT_DIR / "combinacoes_protetoras.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, max(4, len(combo_df)*0.8)))
    cores_c = ["#2ca02c" if v < -2 else "#d62728" if v > 2 else "steelblue"
               for v in combo_df["delta_vs_media"]]
    ax.barh(combo_df["combinacao"][::-1], combo_df["ret_pct"][::-1], color=cores_c[::-1])
    ax.axvline(ref, color="black", ls="--", lw=1.5, label=f"Média ({ref:.1f}%)")
    ax.set_xlabel("Taxa de retorno 30d (%)")
    ax.set_title("Combinações de Condições Protetoras\n(Pacientes COM Documentação Clínica)")
    ax.legend(fontsize=9); ax.grid(axis="x", alpha=0.3)
    for i,(_, row) in enumerate(combo_df[::-1].reset_index(drop=True).iterrows()):
        ax.text(row["ret_pct"]+0.2, i,
                f"n={int(row['n']):,}  Δ={row['delta_vs_media']:+.1f}pp",
                va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "04_combinacoes_protetoras.png", dpi=150)
    plt.show(); plt.close()

## 6. Resumo — Condições de Menor Risco

In [ ]:
print("\n" + "="*65)
print("RESUMO — CONDIÇÕES DE MENOR RISCO EM PACIENTES DOCUMENTADAS")
print("="*65)
print(f"\nBase: {len(df_doc):,} pacientes com documentação clínica")
print(f"Taxa média de retorno 30d neste grupo: {ref:.1f}%\n")

print("PROTETORES IDENTIFICADOS:")
print("  - Condições que reduzem o risco em ≥ 2pp abaixo da média")
print()

protetores_desc = comp[comp["ret_doc_pct"] < ref - 2].sort_values("ret_doc_pct")
for _, row in protetores_desc.iterrows():
    print(f"  [{row['segmento']:20s}] retorno={row['ret_doc_pct']:.1f}%  "
          f"delta={row['delta_pp']:+.1f}pp vs sem-doc")

print()
print("INTERPRETAÇÃO CLÍNICA:")
print("  Pacientes com documentação clínica têm menor risco quando:")
print("  1. CID de gestação normal/ambulatório (não urgência aguda)")
print("  2. Sem red_flag detectado pelo LLM no texto")
print("  3. clinical_status = 'stable' na última evolução")
print("  4. Gestação de termo (IG 37-40s) sem complicação documentada")
print("  5. Manchester verde/amarelo na triagem inicial")
print()
print("  RISCO AUMENTADO mesmo com documentação:")
print("  1. Hemorragia — retorno alto independente de documentação")
print("  2. red_flag presente no texto → qualquer documentação indicativa")
print("  3. severity=high/critical no texto da evolução")

(OUT_DIR / "resumo_menor_risco.txt").write_text(
    f"Análise: menor risco em documentadas\n"
    f"Data: {datetime.now().isoformat()}\n"
    f"N documentadas: {len(df_doc):,}\n"
    f"Taxa retorno média: {ref:.1f}%\n",
    encoding="utf-8"
)

print(f"\n✅ Concluído — {OUT_DIR}")
print(f"   doc_vs_nao_doc_por_segmento.csv")
print(f"   perfil_protetor_documentadas.csv")
print(f"   shap_protetor_risco.csv")
print(f"   combinacoes_protetoras.csv")
print(f"   plots/ (4 figuras)")